In [1]:
import logging
import random
import shutil
from pathlib import Path

import pandas as pd

In [2]:
SOURCE_DIR = "/kaggle/input/datasets/lfreedom2750/mcvsld"          # thư mục gốc chứa các folder class
OUTPUT_DIR = "/kaggle/working/data_split"       # thư mục output sau khi chia

TRAIN_RATIO = 0.8
VAL_RATIO   = 0.1
TEST_RATIO  = 0.1

SEED        = 42
IMAGE_EXTS  = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
COPY_FILES  = True              # True: copy, False: move
SAVE_CSV    = True              # xuất file split_metadata.csv
FORCE_OVERWRITE = True          # notebook nên đặt rõ để tránh hỏi input
DRY_RUN     = False

In [3]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)

In [4]:
def is_image_file(file_path: Path) -> bool:
    return file_path.suffix.lower() in IMAGE_EXTS


def validate_ratios(train: float, val: float, test: float) -> None:
    for name, value in [("TRAIN_RATIO", train), ("VAL_RATIO", val), ("TEST_RATIO", test)]:
        if not (0.0 <= value <= 1.0):
            raise ValueError(f"{name} phải nằm trong [0, 1], nhận được: {value}")
    total = train + val + test
    if abs(total - 1.0) > 1e-8:
        raise ValueError(
            f"Tổng TRAIN_RATIO + VAL_RATIO + TEST_RATIO phải bằng 1.0, "
            f"nhận được: {total:.6f}"
        )


def get_class_names(source_dir: Path) -> list[str]:
    classes = sorted(d.name for d in source_dir.iterdir() if d.is_dir())
    return classes


def make_output_structure(output_dir: Path, class_names: list[str]) -> None:
    for split in ("train", "val", "test"):
        for class_name in class_names:
            (output_dir / split / class_name).mkdir(parents=True, exist_ok=True)


def prepare_output_dir(output_dir: Path, force_overwrite: bool) -> None:
    if not output_dir.exists():
        return

    if force_overwrite:
        logger.warning("Output dir '%s' đã tồn tại → xóa và tạo lại.", output_dir)
        shutil.rmtree(output_dir)
    else:
        raise SystemExit(
            f"Output dir '{output_dir}' đã tồn tại. "
            "Hãy đặt FORCE_OVERWRITE=True hoặc đổi OUTPUT_DIR."
        )

In [5]:
def split_one_class(
    file_list: list[str],
    train_ratio: float = 0.8,
    val_ratio: float   = 0.1,
    test_ratio: float  = 0.1,
) -> tuple[list[str], list[str], list[str]]:
    """Chia file list thành train / val / test, xử lý đúng mọi kích thước."""
    shuffled_files = file_list.copy()
    random.shuffle(shuffled_files)
    n = len(shuffled_files)

    if n == 0:
        return [], [], []

    if n == 1:
        return shuffled_files, [], []

    if n == 2:
        return [shuffled_files[0]], [shuffled_files[1]], []

    train_count = int(n * train_ratio)
    val_count   = int(n * val_ratio)
    test_count  = n - train_count - val_count

    if val_ratio > 0 and val_count == 0:
        val_count += 1
        train_count -= 1
    if test_ratio > 0 and test_count == 0:
        test_count += 1
        train_count -= 1

    train_count = max(train_count, 0)
    val_count   = max(val_count, 0)
    test_count  = max(test_count, 0)

    diff = n - (train_count + val_count + test_count)
    train_count += diff

    return (
        shuffled_files[:train_count],
        shuffled_files[train_count: train_count + val_count],
        shuffled_files[train_count + val_count:],
    )


def transfer_files(
    files: list[str],
    src_dir: Path,
    dst_dir: Path,
    copy_files: bool = True,
) -> None:
    for file_name in files:
        src = src_dir / file_name
        dst = dst_dir / file_name
        if copy_files:
            shutil.copy2(src, dst)
        else:
            shutil.move(str(src), str(dst))

In [6]:
def print_summary(summary: dict[str, dict]) -> None:
    logger.info("========== SPLIT SUMMARY ==========")
    for class_name, s in summary.items():
        logger.info(
            "%-14s total=%4d | train=%4d | val=%4d | test=%4d",
            class_name, s["total"], s["train"], s["val"], s["test"],
        )
    logger.info("====================================")

In [7]:
validate_ratios(TRAIN_RATIO, VAL_RATIO, TEST_RATIO)
random.seed(SEED)

source_dir = Path(SOURCE_DIR)
output_dir = Path(OUTPUT_DIR)

if not source_dir.exists():
    raise FileNotFoundError(f"Không tìm thấy SOURCE_DIR: {source_dir}")

if source_dir.resolve() == output_dir.resolve():
    raise ValueError("SOURCE_DIR và OUTPUT_DIR không được trùng nhau.")

class_names = get_class_names(source_dir)
if not class_names:
    raise ValueError("Không tìm thấy folder class nào trong SOURCE_DIR.")

logger.info("Classes found: %s", class_names)
class_names

04:32:37 [INFO] Classes found: ['Chickenpox', 'Cowpox', 'HFMD', 'Healthy', 'Measles', 'Monkeypox']


['Chickenpox', 'Cowpox', 'HFMD', 'Healthy', 'Measles', 'Monkeypox']

In [8]:
initial_stats = []

for class_name in class_names:
    class_dir = source_dir / class_name
    image_files = sorted(
        f.name for f in class_dir.iterdir()
        if f.is_file() and is_image_file(f)
    )
    initial_stats.append({
        "label": class_name,
        "total_images": len(image_files)
    })

initial_df = pd.DataFrame(initial_stats)
initial_df

,label,total_images
0,Chickenpox,900
1,Cowpox,792
2,HFMD,1932
3,Healthy,1368
4,Measles,660
5,Monkeypox,3408


In [9]:
if DRY_RUN:
    logger.info("[DRY-RUN] Chỉ hiển thị kế hoạch, không copy/move file.")
else:
    prepare_output_dir(output_dir, FORCE_OVERWRITE)
    make_output_structure(output_dir, class_names)

if not COPY_FILES and not DRY_RUN:
    logger.warning(
        "COPY_FILES=False: file gốc sẽ bị di chuyển ra khỏi '%s'. "
        "Hãy đảm bảo bạn đã backup trước!",
        source_dir,
    )

In [10]:
records = []
summary = {}

for class_name in class_names:
    class_dir = source_dir / class_name
    image_files = sorted(
        f.name for f in class_dir.iterdir()
        if f.is_file() and is_image_file(f)
    )

    train_files, val_files, test_files = split_one_class(
        image_files,
        TRAIN_RATIO,
        VAL_RATIO,
        TEST_RATIO
    )

    summary[class_name] = {
        "total": len(image_files),
        "train": len(train_files),
        "val":   len(val_files),
        "test":  len(test_files),
    }

    if not DRY_RUN:
        transfer_files(train_files, class_dir, output_dir / "train" / class_name, COPY_FILES)
        transfer_files(val_files,   class_dir, output_dir / "val"   / class_name, COPY_FILES)
        transfer_files(test_files,  class_dir, output_dir / "test"  / class_name, COPY_FILES)

    for split_name, split_files in (
        ("train", train_files),
        ("val",   val_files),
        ("test",  test_files),
    ):
        for fname in split_files:
            records.append({
                "file_name": fname,
                "label":     class_name,
                "split":     split_name,
                "path":      str(output_dir / split_name / class_name / fname),
            })

print_summary(summary)

04:33:37 [INFO] ========== SPLIT SUMMARY ==========
04:33:37 [INFO] Chickenpox     total= 900 | train= 720 | val=  90 | test=  90
04:33:37 [INFO] Cowpox         total= 792 | train= 633 | val=  79 | test=  80
04:33:37 [INFO] HFMD           total=1932 | train=1545 | val= 193 | test= 194
04:33:37 [INFO] Healthy        total=1368 | train=1094 | val= 136 | test= 138
04:33:37 [INFO] Measles        total= 660 | train= 528 | val=  66 | test=  66
04:33:37 [INFO] Monkeypox      total=3408 | train=2726 | val= 340 | test= 342
04:33:37 [INFO] ====================================


In [11]:
summary_df = pd.DataFrame(summary).T.reset_index().rename(columns={"index": "label"})
summary_df

,label,total,train,val,test
0,Chickenpox,900,720,90,90
1,Cowpox,792,633,79,80
2,HFMD,1932,1545,193,194
3,Healthy,1368,1094,136,138
4,Measles,660,528,66,66
5,Monkeypox,3408,2726,340,342


In [12]:
if SAVE_CSV and not DRY_RUN:
    df = pd.DataFrame(records)
    csv_path = output_dir / "split_metadata.csv"
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    logger.info("Đã lưu file metadata: %s", csv_path)
    df.head()
else:
    pd.DataFrame(records).head()

04:33:37 [INFO] Đã lưu file metadata: /kaggle/working/data_split/split_metadata.csv


In [13]:
verify_rows = []

if not DRY_RUN:
    for split_name in ["train", "val", "test"]:
        for class_name in class_names:
            split_class_dir = output_dir / split_name / class_name
            num_files = len([
                f for f in split_class_dir.iterdir()
                if f.is_file() and is_image_file(f)
            ])
            verify_rows.append({
                "split": split_name,
                "label": class_name,
                "num_files": num_files
            })

    verify_df = pd.DataFrame(verify_rows)
    verify_df
else:
    print("DRY_RUN=True nên không có thư mục output để verify.")

In [14]:
if not DRY_RUN:
    pivot_df = verify_df.pivot(index="label", columns="split", values="num_files").fillna(0).astype(int)
    pivot_df